# Mi recuperación de intentos del pipeline

Uso este notebook manual cuando un intento quedó sin cierre y ya no hay ejecuciones activas.
El modo preview solo comprueba evidencia.
El modo apply inserta decisiones en audit_gtfs_attempt_resolutions.
No modifica los intentos originales ni las tablas Raw, Bronze o Silver.
El planificador muestra después las resoluciones como RESOLVED.
No añado este notebook a un Job automático.


In [ ]:
# BLOQUE 1 — Repositorio y solicitudes explícitas
from pathlib import Path
import sys
import json

defaults = {"repo_root": "", "catalog": "workspace", "schema": "gtfs_bogota",
            "resolutions_json": "[]", "mode": "preview", "confirm_executions_stopped": "false"}
for key, default in defaults.items(): dbutils.widgets.text(key, default)
parameters = {key: dbutils.widgets.get(key).strip() for key in defaults}
root_parameter = parameters.pop("repo_root")
if root_parameter and not Path(root_parameter).is_absolute(): raise ValueError("repo_root debe ser absoluto.")
current = Path.cwd().resolve()
roots = [Path(root_parameter)] if root_parameter else [current, *current.parents]
root = next((p for p in roots if (p / "databricks/gtfs_silver/planner_recovery.py").is_file()), None)
if root is None: raise RuntimeError("Sincroniza el repositorio completo y configura repo_root.")
module_path = str(root / "databricks")
if module_path not in sys.path: sys.path.insert(0, module_path)
from gtfs_silver.planner_recovery import resolve_attempt
requests = json.loads(parameters.pop("resolutions_json"))
if not isinstance(requests, list) or not 1 <= len(requests) <= 20:
    raise ValueError("resolutions_json debe contener entre 1 y 20 solicitudes explícitas.")
mode = parameters.pop("mode")
confirmation = parameters.pop("confirm_executions_stopped").lower()
if mode not in {"preview", "apply"} or confirmation not in {"true", "false"}:
    raise ValueError("Revisa mode y confirm_executions_stopped.")


In [ ]:
# BLOQUE 2 — Comprobación de todos los casos antes de escribir
previews = [resolve_attempt(spark, **parameters, **request, mode="preview") for request in requests]
print(json.dumps(previews, ensure_ascii=False, indent=2))
print("Vista previa completa: no escribí resoluciones ni modifiqué tablas de datos.")


In [ ]:
# BLOQUE 3 — Resolución manual con evidencia recién comprobada
if mode == "apply":
    if confirmation != "true":
        raise ValueError("Para aplicar, confirma que las ejecuciones terminaron o se detuvieron.")
    results = []
    for request in requests:
        result = resolve_attempt(spark, **parameters, **request, mode="apply", executions_stopped=True)
        results.append(result)
        print(json.dumps(result, ensure_ascii=False, indent=2))
    print("Resoluciones registradas. Ejecuta notebook 11 nuevamente en vista previa.")
else:
    print("Modo preview: no registré resoluciones.")
